# 德译英机器翻译 - Transformer 演示

本 notebook 展示如何使用标准 Transformer 模型进行德语到英语的机器翻译。
模型基于 "Attention Is All You Need" 论文实现，使用 Multi30k 数据集训练。

## 1. 环境设置

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

## 2. 配置参数

In [ ]:
# 模型配置 (基于论文的 base 配置)
config = {
    'data_dir': './transformer/data',
    'checkpoint_dir': './transformer/checkpoints',
    
    # 模型参数
    'd_model': 512,
    'nhead': 8,
    'd_ff': 2048,
    'num_encoder_layers': 6,
    'num_decoder_layers': 6,
    'dropout': 0.1,
    'max_len': 128,
    
    # 训练参数
    'batch_size': 64,
    'epochs': 20,
    'lr': 1.0,
    'weight_decay': 1e-4,
    'warmup_steps': 4000,
    'grad_clip_norm': 1.0,
    'early_stopping_patience': 5,
    'num_workers': 2,
}

print('Configuration:')
for k, v in config.items():
    print(f'  {k}: {v}')

## 3. 数据加载

In [ ]:
from transformer.data import get_dataloaders, Vocabulary

# 加载数据
train_loader, val_loader, src_vocab, tgt_vocab = get_dataloaders(
    data_dir=config['data_dir'],
    batch_size=config['batch_size'],
    num_workers=config['num_workers'],
    max_len=config['max_len'],
)

print(f'\n源语言词汇表大小 (德语): {len(src_vocab)}')
print(f'目标语言词汇表大小 (英语): {len(tgt_vocab)}')
print(f'训练集批次数: {len(train_loader)}')
print(f'验证集批次数: {len(val_loader)}')

In [ ]:
# 展示样本
def show_samples(loader, src_vocab, tgt_vocab, num_samples=5):
    batch = next(iter(loader))
    src = batch['src']
    tgt = batch['tgt']
    
    print('样本展示:')
    print('-' * 80)
    for i in range(min(num_samples, src.size(0))):
        src_text = src_vocab.decode(src[i].tolist())
        tgt_text = tgt_vocab.decode(tgt[i].tolist())
        print(f'德语: {src_text}')
        print(f'英语: {tgt_text}')
        print('-' * 80)

show_samples(train_loader, src_vocab, tgt_vocab)

## 4. 模型定义

In [ ]:
from transformer.model import Transformer

# 创建模型
model = Transformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=config['d_model'],
    nhead=config['nhead'],
    d_ff=config['d_ff'],
    num_encoder_layers=config['num_encoder_layers'],
    num_decoder_layers=config['num_decoder_layers'],
    dropout=config['dropout'],
    max_len=config['max_len'],
).to(device)

# 打印模型结构
print(model)
print(f'\n模型参数量: {sum(p.numel() for p in model.parameters()):,}')

## 5. 训练

In [ ]:
from transformer.trainer import WarmupLRScheduler, LabelSmoothingLoss, train_epoch, evaluate

# 损失函数和优化器
criterion = LabelSmoothingLoss(
    vocab_size=len(tgt_vocab),
    padding_idx=tgt_vocab.pad_idx,
    smoothing=0.1
)

optimizer = optim.Adam(
    model.parameters(),
    lr=config['lr'],
    betas=(0.9, 0.98),
    eps=1e-9,
    weight_decay=config['weight_decay']
)

# 学习率调度器
scheduler = WarmupLRScheduler(
    optimizer,
    d_model=config['d_model'],
    warmup_steps=config['warmup_steps']
)

# 记录训练历史
history = {
    'train_loss': [],
    'val_loss': [],
    'lr': []
}

# Early stopping
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

In [ ]:
# 训练循环
print('开始训练...')
step = 0

for epoch in range(config['epochs']):
    # 训练
    train_loss, _ = train_epoch(
        model, train_loader, criterion, optimizer, scheduler,
        device, src_vocab.pad_idx, config['grad_clip_norm']
    )
    
    # 验证
    val_loss, _ = evaluate(
        model, val_loader, criterion, device, src_vocab.pad_idx
    )
    
    # 记录
    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(current_lr)
    
    print(f'Epoch {epoch+1}/{config["epochs"]}: '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'LR: {current_lr:.6f}')
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= config['early_stopping_patience']:
            print(f'Early stopping at epoch {epoch+1}')
            break

# 加载最佳模型
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f'\n加载最佳模型 (Val Loss: {best_val_loss:.4f})')

## 6. 训练曲线可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss 曲线
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# 学习率曲线
axes[1].plot(history['lr'], label='Learning Rate', marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate Schedule')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. 交互式演示 - 德译英翻译

输入德语句子，模型将翻译成英语。

In [ ]:
from transformer.trainer import greedy_decode

def translate(sentence: str) -> str:
    """翻译德语句子到英语"""
    model.eval()
    
    # 编码源句子
    src_indices = src_vocab.encode(sentence)
    src_tensor = torch.tensor([src_indices], dtype=torch.long, device=device)
    
    # 贪婪解码
    with torch.no_grad():
        output_indices = greedy_decode(
            model,
            src_tensor,
            max_len=config['max_len'],
            start_symbol=tgt_vocab.sos_idx,
            end_symbol=tgt_vocab.eos_idx,
            pad_idx=src_vocab.pad_idx,
            device=device
        )
    
    # 解码目标句子
    translation = tgt_vocab.decode(output_indices[0].tolist())
    return translation

# 测试翻译
test_sentences = [
    "ein mann steht auf einer straße",
    "eine frau liest ein buch",
    "zwei kinder spielen im park",
]

print('翻译测试:')
print('-' * 60)
for sent in test_sentences:
    translation = translate(sent)
    print(f'德语: {sent}')
    print(f'英语: {translation}')
    print('-' * 60)

In [ ]:
# 交互式翻译界面
input_text = widgets.Text(
    value='',
    placeholder='输入德语句子...',
    description='德语:',
    layout=widgets.Layout(width='500px')
)

translate_btn = widgets.Button(
    description='翻译',
    button_style='success',
    layout=widgets.Layout(width='100px')
)

output_area = widgets.Output()

# 示例按钮
example_btns = []
example_sentences = [
    "ein mann steht auf einer straße",
    "eine frau liest ein buch",
    "zwei kinder spielen im park",
    "der hund läuft im garten",
]

def on_example_click(btn):
    input_text.value = btn.description

for sent in example_sentences:
    btn = widgets.Button(
        description=sent,
        layout=widgets.Layout(width='auto')
    )
    btn.on_click(on_example_click)
    example_btns.append(btn)

def on_translate_click(btn):
    with output_area:
        clear_output()
        if input_text.value.strip():
            translation = translate(input_text.value)
            print(f'\n德语: {input_text.value}')
            print(f'英语: {translation}')
        else:
            print('请输入德语句子')

translate_btn.on_click(on_translate_click)

# 显示界面
display(widgets.HTML('<b>德译英翻译演示</b>'))
display(widgets.HTML('<p>点击示例或输入德语句子:</p>'))
display(widgets.HBox(example_btns))
display(widgets.HBox([input_text, translate_btn]))
display(output_area)

## 说明

1. **模型架构**: 标准 Transformer 编码器-解码器架构，遵循 "Attention Is All You Need" 论文
2. **训练**: 使用 Adam 优化器 + Warmup 学习率调度 + 标签平滑损失
3. **推理**: 使用贪婪解码生成翻译
4. **数据**: Multi30k 德语-英语数据集